# Derin Öğrenme ile Saldırı Tespit Sistemi (IDS)

Bu notebook, NSL-KDD veri setini kullanarak çeşitli derin öğrenme modelleri (DNN, ResNet, LSTM, Attention, CNN) ile saldırı tespiti yapmayı amaçlar.

**İçerik:**
1. Kütüphanelerin Yüklenmesi
2. Veri Seti Hazırlığı (NSL-KDD)
3. Model Mimarileri
4. Eğitim ve Değerlendirme Yardımcıları
5. Eğitim Döngüsü

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm  # Notebook için tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# Grafik ayarları
plt.style.use('ggplot')
sns.set_palette('husl')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan Cihaz: {device}')

## 1. Veri Seti Hazırlama ve Ön İşleme
NSL-KDD veri seti yüklenir, temizlenir ve PyTorch DataLoader formatına getirilir.

In [ ]:

# NSL-KDD Sütun İsimleri (Referans için)
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label'
]

class KDDDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def balance_dataset(df, target_col='label', method='undersample', ratio=0.4):
    """Veri setini dengeler"""
    print(f"\n⚖️ VERİ DENGELENİYOR (Method: {method}, Ratio: {ratio})")
    
    # Sınıfları ayır
    counts = df[target_col].value_counts()
    if len(counts) < 2:
        return df
        
    majority_class = counts.index[0]
    minority_class = counts.index[1]
    
    df_majority = df[df[target_col] == majority_class]
    df_minority = df[df[target_col] == minority_class]
    
    n_minority = len(df_minority)
    n_majority = len(df_majority)
    
    if method == 'undersample':
        target_majority = int(n_minority / ratio)
        df_majority_sampled = resample(df_majority, replace=False, 
                                      n_samples=min(target_majority, n_majority), random_state=42)
        df_balanced = pd.concat([df_majority_sampled, df_minority])
    elif method == 'oversample':
        target_minority = int(n_majority * ratio)
        df_minority_sampled = resample(df_minority, replace=True, 
                                      n_samples=target_minority, random_state=42)
        df_balanced = pd.concat([df_majority, df_minority_sampled])
    else:
        # Combined mantığı
        target_size = int((n_majority + n_minority) * ratio)
        df_majority_sampled = resample(df_majority, replace=False, n_samples=min(target_size, n_majority), random_state=42)
        df_minority_sampled = resample(df_minority, replace=True, n_samples=target_size, random_state=42)
        df_balanced = pd.concat([df_majority_sampled, df_minority_sampled])
    
    return df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

def load_and_preprocess_data(train_path='KDDTrain+.txt', test_path='KDDTest+.txt', 
                            binary=True, balance_train=True, balance_test=False,
                            balance_method='combined', balance_ratio=0.4):
    """KDD Cup 99 verisini yükler ve hatalı etiketlemeyi düzeltir"""
    
    print("="*80 + "\n📊 VERİ YÜKLEME VE DÜZELTME BAŞLADI\n" + "="*80)
    
    # Veriyi yükle
    train_df = pd.read_csv(train_path, header=None, low_memory=False)
    test_df = pd.read_csv(test_path, header=None, low_memory=False)
    
    # KRİTİK DÜZELTME: NSL-KDD'de etiket 41. sütundadır. 42. sütun (zorluk) atılmalıdır.
    # Senin eski kodun shape[1]-1 alıyordu, bu da zorluk seviyesini etiket sanıyordu.
    label_idx = 41 
    
    def process_labels(df):
        # Etiketi string yap, temizle ve içinde 'normal' geçip geçmediğine bak
        s = df[label_idx].astype(str).str.lower().str.strip()
        return s.apply(lambda x: 'normal' if 'normal' in x else 'attack')

    train_df['label'] = process_labels(train_df)
    test_df['label'] = process_labels(test_df)
    
    # Gereksiz olan eski label ve zorluk sütunlarını düşür (Özelliklerden ayır)
    # NSL-KDD'de ilk 41 sütun (0-40 arası) asıl özelliklerdir.
    X_train_raw = train_df.iloc[:, :41]
    X_test_raw = test_df.iloc[:, :41]
    
    # Kategorik sütunları (1, 2, 3. sütunlar: protocol, service, flag) işle
    categorical_cols = [1, 2, 3]
    for col in categorical_cols:
        le_cat = LabelEncoder()
        X_train_raw.iloc[:, col] = le_cat.fit_transform(X_train_raw.iloc[:, col].astype(str))
        # Test setinde eğitimde olmayan bir değer gelirse hata vermemesi için:
        X_test_raw.iloc[:, col] = X_test_raw.iloc[:, col].astype(str).map(
            lambda x: x if x in le_cat.classes_ else le_cat.classes_[0])
        X_test_raw.iloc[:, col] = le_cat.transform(X_test_raw.iloc[:, col])

    # Dengeleme işlemi için geçici dataframe oluştur
    train_proc = X_train_raw.copy()
    train_proc['label'] = train_df['label']
    test_proc = X_test_raw.copy()
    test_proc['label'] = test_df['label']

    if balance_train:
        train_proc = balance_dataset(train_proc, method=balance_method, ratio=balance_ratio)
    
    if balance_test:
        test_proc = balance_dataset(test_proc, method=balance_method, ratio=balance_ratio)

    # Nihai özellikler ve etiketler
    X_train = train_proc.drop('label', axis=1).values
    y_train_str = train_proc['label'].values
    X_test = test_proc.drop('label', axis=1).values
    y_test_str = test_proc['label'].values

    # Ana Label Encoder (Normal/Attack -> 0/1)
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_str)
    y_test = le.transform(y_test_str)
    class_names = le.classes_

    # Normalizasyon
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    X_train = np.nan_to_num(X_train)
    X_test = np.nan_to_num(X_test)

    # Scaler kaydet
    with open('scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
        
    print(f"✅ VERİ HAZIR! Train: {len(X_train)}, Test: {len(X_test)}")
    print(f"📊 Sınıf Dağılımı (Test): {pd.Series(y_test_str).value_counts().to_dict()}")
    
    return X_train, X_test, y_train, y_test, scaler, le, class_names

def create_dataloaders(X_train, X_test, y_train, y_test, batch_size=256):
    train_dataset = KDDDataset(X_train, y_train)
    test_dataset = KDDDataset(X_test, y_test)
    return DataLoader(train_dataset, batch_size=batch_size, shuffle=True), \
           DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 2. Derin Öğrenme Modelleri
Burada 5 farklı model mimarisi tanımlanmıştır:
- **SimpleDNN**: Temel Tam Bağlantılı Ağ
- **ResNetIDS**: Residual Bağlantılı Ağ
- **LSTMBasedIDS**: Zaman Serisi/Sıralı Veri Yaklaşımı
- **AttentionIDS**: Self-Attention Mekanizması
- **CNN1DIDS**: 1 Boyutlu Konvolüsyonel Ağ

In [ ]:


class SimpleDNN(nn.Module):
    """Basit Deep Neural Network - Baseline Model"""
    def __init__(self, input_size, num_classes, hidden_sizes=[128, 64]):
        super(SimpleDNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


class ResidualBlock(nn.Module):
    """Residual Block for ResNet-like architecture"""
    def __init__(self, in_features, out_features):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)
        self.fc2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)
        
        # Shortcut connection
        self.shortcut = nn.Linear(in_features, out_features) if in_features != out_features else nn.Identity()
        
    def forward(self, x):
        residual = self.shortcut(x)
        
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.bn2(self.fc2(out))
        out += residual
        out = F.relu(out)
        
        return out


class ResNetIDS(nn.Module):
    """ResNet-like architecture for IDS - En Güçlü Model"""
    def __init__(self, input_size, num_classes, hidden_size=128):
        super(ResNetIDS, self).__init__()
        
        self.input_layer = nn.Linear(input_size, hidden_size)
        
        self.res_block1 = ResidualBlock(hidden_size, hidden_size)
        self.res_block2 = ResidualBlock(hidden_size, hidden_size)
        self.res_block3 = ResidualBlock(hidden_size, hidden_size // 2)
        
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Linear(hidden_size // 2, num_classes)
        
    def forward(self, x):
        x = F.relu(self.input_layer(x))
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.res_block3(x)
        x = self.dropout(x)
        x = self.fc_out(x)
        return x


class LSTMBasedIDS(nn.Module):
    """LSTM-based IDS model"""
    def __init__(self, input_size, num_classes, hidden_size=128, num_layers=2):
        super(LSTMBasedIDS, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM katmanı
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=0.3 if num_layers > 1 else 0)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size // 2, num_classes)
        
    def forward(self, x):
        # x shape: (batch, features)
        # LSTM bekler: (batch, seq_len, features)
        x = x.unsqueeze(1)  # (batch, 1, features)
        
        # LSTM
        lstm_out, _ = self.lstm(x)
        
        # Son çıktıyı al
        x = lstm_out[:, -1, :]  # (batch, hidden_size)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x


class AttentionIDS(nn.Module):
    """Self-Attention based IDS model"""
    def __init__(self, input_size, num_classes, hidden_size=128, num_heads=4):
        super(AttentionIDS, self).__init__()
        
        self.embedding = nn.Linear(input_size, hidden_size)
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        
        self.layer_norm1 = nn.LayerNorm(hidden_size)
        self.layer_norm2 = nn.LayerNorm(hidden_size)
        
        # Feed-forward network
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size * 2, hidden_size)
        )
        
        self.dropout = nn.Dropout(0.3)
        self.fc_out = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # Embedding
        x = self.embedding(x)  # (batch, hidden_size)
        x = x.unsqueeze(1)  # (batch, 1, hidden_size)
        
        # Self-attention
        attn_out, _ = self.attention(x, x, x)
        x = self.layer_norm1(x + attn_out)
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.layer_norm2(x + ff_out)
        
        # Output
        x = x.squeeze(1)  # (batch, hidden_size)
        x = self.dropout(x)
        x = self.fc_out(x)
        
        return x


class CNN1DIDS(nn.Module):
    """1D CNN for IDS"""
    def __init__(self, input_size, num_classes, num_filters=64):
        super(CNN1DIDS, self).__init__()
        
        self.conv1 = nn.Conv1d(1, num_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(num_filters)
        
        self.conv2 = nn.Conv1d(num_filters, num_filters * 2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_filters * 2)
        
        self.conv3 = nn.Conv1d(num_filters * 2, num_filters * 4, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(num_filters * 4)
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(num_filters * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        # x: (batch, features)
        x = x.unsqueeze(1)  # (batch, 1, features)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.pool(x)  # (batch, filters*4, 1)
        x = x.squeeze(-1)  # (batch, filters*4)
        
        x = self.fc(x)
        
        return x


def get_model(model_name, input_size, num_classes, **kwargs):
    """Model seçim fonksiyonu"""
    
    models = {
        'simple_dnn': SimpleDNN,
        'resnet_ids': ResNetIDS,
        'lstm_ids': LSTMBasedIDS,
        'attention_ids': AttentionIDS,
        'cnn1d_ids': CNN1DIDS
    }
    
    if model_name not in models:
        raise ValueError(f"Model '{model_name}' bulunamadı. Mevcut: {list(models.keys())}")
    
    return models[model_name](input_size, num_classes, **kwargs)


if __name__ == "__main__":
    # Test
    input_size = 43  # Sizin veri setinizde 43 özellik var
    num_classes = 2  # Binary: normal vs attack
    batch_size = 32
    
    x = torch.randn(batch_size, input_size)
    
    print("🧪 Model Testleri:\n")
    
    models = {
        'SimpleDNN': SimpleDNN(input_size, num_classes),
        'ResNetIDS': ResNetIDS(input_size, num_classes),
        'LSTMIDS': LSTMBasedIDS(input_size, num_classes),
        'AttentionIDS': AttentionIDS(input_size, num_classes),
        'CNN1DIDS': CNN1DIDS(input_size, num_classes)
    }
    
    for name, model in models.items():
        output = model(x)
        params = sum(p.numel() for p in model.parameters())
        print(f"✓ {name}")
        print(f"  Output: {output.shape}")
        print(f"  Parametreler: {params:,}\n")

## 3. Eğitim ve Değerlendirme Sınıfları
`Trainer` sınıfı eğitim döngüsünü, validasyonu ve model kaydetmeyi yönetir. `FocalLoss` dengesiz veri setleri için kullanılır.

In [ ]:



class FocalLoss(nn.Module):
    """
    Focal Loss for imbalanced classification
    Zor örneklere daha fazla odaklanır
    """
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Class weights
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


class Trainer:
    def __init__(self, model, device, class_names, class_weights=None, use_focal_loss=False):
        self.model = model.to(device)
        self.device = device
        self.class_names = class_names
        self.class_weights = class_weights
        self.use_focal_loss = use_focal_loss
        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        
    def train_epoch(self, train_loader, criterion, optimizer):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc='Training', leave=False)
        for X_batch, y_batch in pbar:
            X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
            
            optimizer.zero_grad()
            outputs = self.model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
            
            pbar.set_postfix({'loss': f'{total_loss/(pbar.n+1):.4f}', 
                            'acc': f'{100.*correct/total:.2f}%'})
        
        return total_loss / len(train_loader), 100. * correct / total
    
    def evaluate(self, test_loader, criterion):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for X_batch, y_batch in tqdm(test_loader, desc='Evaluating', leave=False):
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += y_batch.size(0)
                correct += predicted.eq(y_batch).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())
        
        accuracy = 100. * correct / total
        avg_loss = total_loss / len(test_loader)
        
        return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)
    
    def fit(self, train_loader, test_loader, epochs=30, lr=0.001, 
            weight_decay=1e-4, patience=10, save_path='best_model.pt'):
        
        # Loss function seçimi
        if self.use_focal_loss:
            print("📊 Loss Function: Focal Loss")
            criterion = FocalLoss(alpha=self.class_weights, gamma=2)
        else:
            print("📊 Loss Function: CrossEntropyLoss with Class Weights")
            criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
        
        best_acc = 0
        patience_counter = 0
        
        print(f"\n{'='*80}")
        print(f"🚀 EĞİTİM BAŞLADI: {epochs} epoch")
        print(f"{'='*80}\n")
        
        for epoch in range(epochs):
            print(f"Epoch {epoch+1}/{epochs}")
            
            train_loss, train_acc = self.train_epoch(train_loader, criterion, optimizer)
            val_loss, val_acc, _, _ = self.evaluate(test_loader, criterion)
            
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            
            # Learning rate güncelle
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_acc)
            new_lr = optimizer.param_groups[0]['lr']
            
            if old_lr != new_lr:
                print(f"  📉 Learning rate düşürüldü: {old_lr:.6f} -> {new_lr:.6f}")
            
            print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
            
            # Early stopping ve model kaydetme
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(self.model.state_dict(), save_path)
                print(f"  ✓ Model kaydedildi (Acc: {best_acc:.2f}%)")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"\n⚠️ Early stopping! {patience} epoch boyunca iyileşme yok.")
                    break
            print()
        
        # En iyi modeli yükle
        self.model.load_state_dict(torch.load(save_path))
        print(f"\n{'='*80}")
        print(f"✅ EĞİTİM TAMAMLANDI! En iyi doğruluk: {best_acc:.2f}%")
        print(f"{'='*80}\n")
        
        return best_acc
    
    def detailed_evaluation(self, test_loader, save_dir='results'):
        """Detaylı değerlendirme ve sonuçları kaydetme"""
        
        os.makedirs(save_dir, exist_ok=True)
        
        if self.use_focal_loss:
            criterion = FocalLoss(alpha=self.class_weights, gamma=2)
        else:
            criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        
        val_loss, val_acc, y_pred, y_true = self.evaluate(test_loader, criterion)
        
        # Classification report
        report = classification_report(y_true, y_pred, target_names=self.class_names, 
                                      output_dict=True, zero_division=0)
        
        print("\n" + "="*80)
        print("📊 DETAYLI SINIFLANDIRMA RAPORU")
        print("="*80 + "\n")
        print(classification_report(y_true, y_pred, target_names=self.class_names, zero_division=0))
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=self.class_names, yticklabels=self.class_names)
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(f'{save_dir}/confusion_matrix.png', dpi=300)
        plt.close()
        
        # Sınıf bazlı metrikler
        per_class_metrics = {}
        for i, class_name in enumerate(self.class_names):
            per_class_metrics[class_name] = {
                'precision': report[class_name]['precision'],
                'recall': report[class_name]['recall'],
                'f1-score': report[class_name]['f1-score'],
                'support': report[class_name]['support']
            }
        
        # Sonuçları kaydet
        results = {
            'overall_accuracy': val_acc,
            'overall_loss': val_loss,
            'macro_avg': report['macro avg'],
            'weighted_avg': report['weighted avg'],
            'per_class_metrics': per_class_metrics
        }
        
        with open(f'{save_dir}/evaluation_results.json', 'w') as f:
            json.dump(results, f, indent=4)
        
        print(f"\n✓ Sonuçlar kaydedildi: {save_dir}/")
        
        return results
    
    def plot_history(self, save_path='training_history.png'):
        """Eğitim geçmişini görselleştir"""
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(self.history['train_loss'], label='Train Loss', marker='o')
        ax1.plot(self.history['val_loss'], label='Val Loss', marker='s')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Accuracy
        ax2.plot(self.history['train_acc'], label='Train Accuracy', marker='o')
        ax2.plot(self.history['val_acc'], label='Val Accuracy', marker='s')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.set_title('Training and Validation Accuracy')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()
        
        print(f"✓ Eğitim grafiği: {save_path}")


def calculate_class_weights(y_train, device):
    """Class weights hesapla (imbalanced data için)"""
    from sklearn.utils.class_weight import compute_class_weight
    
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = torch.FloatTensor(weights).to(device)
    
    print(f"\n⚖️ Class Weights:")
    for i, w in enumerate(weights):
        print(f"   Class {i}: {w:.4f}")
    
    return class_weights


def train_single_model(model_name='resnet_ids', epochs=20, batch_size=256, lr=0.001,
                      balance_data=True, use_focal_loss=False, patience=10):
    """Tek bir model eğit"""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n🖥️ Device: {device}")
    
    # Veri yükleme
    print(f"\n📊 Veri yükleniyor...")
    X_train, X_test, y_train, y_test, scaler, le, class_names = load_and_preprocess_data(
        binary=False,
        balance_train=balance_data,
        balance_test=False,  # Test dengelemeyi kapat!
        balance_method='oversample',  # Sadece oversample
        balance_ratio=0.15  # Daha az agresif
    )
    
    train_loader, test_loader = create_dataloaders(X_train, X_test, y_train, y_test, batch_size)
    
    input_size = X_train.shape[1]
    num_classes = len(class_names)
    
    print(f"\n🤖 Model: {model_name}")
    print(f"   Input: {input_size} özellik")
    print(f"   Output: {num_classes} sınıf")
    
    # Class weights hesapla
    class_weights = calculate_class_weights(y_train, device)
    
    # Model oluştur
    model = get_model(model_name, input_size, num_classes)
    
    # Trainer
    trainer = Trainer(model, device, class_names, 
                     class_weights=class_weights,
                     use_focal_loss=use_focal_loss)
    
    # Klasör oluştur
    save_dir = f'results/{model_name}'
    os.makedirs(save_dir, exist_ok=True)
    
    # Eğit
    save_path = f'{save_dir}/{model_name}_best.pt'
    best_acc = trainer.fit(train_loader, test_loader, epochs=epochs, lr=lr, 
                          save_path=save_path, patience=patience)
    
    # Değerlendir
    results = trainer.detailed_evaluation(test_loader, save_dir=save_dir)
    
    # Grafik
    trainer.plot_history(f'{save_dir}/training_history.png')
    
    return trainer, results


def train_all_models(epochs=20, balance_data=True, use_focal_loss=False):
    """Tüm modelleri eğit"""
    
    models_to_train = ['simple_dnn', 'resnet_ids', 'lstm_ids', 'attention_ids', 'cnn1d_ids']
    
    all_results = {}
    
    print(f"\n🎯 Ayarlar:")
    print(f"   Balance Data: {balance_data}")
    print(f"   Focal Loss: {use_focal_loss}")
    print(f"   Epochs: {epochs}\n")
    
    for model_name in models_to_train:
        print(f"\n{'#'*80}")
        print(f"# {model_name.upper()} EĞİTİLİYOR")
        print(f"{'#'*80}\n")
        
        try:
            trainer, results = train_single_model(
                model_name=model_name,
                epochs=epochs,
                batch_size=256,
                lr=0.001,
                balance_data=balance_data,
                use_focal_loss=use_focal_loss
            )
            
            all_results[model_name] = results
            print(f"\n✅ {model_name} TAMAMLANDI! Doğruluk: {results['overall_accuracy']:.2f}%")
            
        except Exception as e:
            print(f"\n❌ {model_name} HATASI: {e}")
            import traceback
            traceback.print_exc()
        
        print(f"\n{'#'*80}\n")
    
    # Özet
    print(f"\n{'='*80}")
    print(f"📊 TÜM MODELLERİN ÖZETİ")
    print(f"{'='*80}\n")
    
    for model_name, results in all_results.items():
        print(f"{model_name:20s}: {results['overall_accuracy']:.2f}%")
    
    print(f"\n{'='*80}\n")
    
    return all_results



## 4. Modellerin Eğitilmesi
Aşağıdaki hücreyi çalıştırarak tüm modelleri sırasıyla eğitebilir veya parametreleri değiştirerek tek bir model eğitebilirsiniz.

In [ ]:
# Çıktı klasörü
os.makedirs('results', exist_ok=True)

# İsterseniz tek bir model eğitin:
# trainer, results = train_single_model(model_name='resnet_ids', epochs=10)

# Veya tüm modelleri eğitin:
all_results = train_all_models(epochs=15, balance_data=True, use_focal_loss=True)